# GlacierMonitor – Part 1: Data Collection
Sentinel-2 (Microsoft Planetary Computer), Copernicus DEM, Landsat thermal. No account needed.

**Run top to bottom in Google Colab.** Outputs go to `MyDrive/GlacierMonitor/raw/`.

Steps: 1) setup → 2) list good dates per year → 3) preview a date → 4) fill in `CHOSEN` → 5) export all years → 6) DEM → 7) Landsat thermal → 8) scene log.

In [ ]:
!pip -q install pystac-client planetary-computer odc-stac rioxarray

In [ ]:
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, os
import pystac_client, planetary_computer, rioxarray
from odc.stac import load

# Save straight to Google Drive (skip these 2 lines if running locally)
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/GlacierMonitor/raw'
for sub in ['sentinel2', 'dem', 'landsat_thermal']:
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace)

# Study area: Khumbu + Ngozumpa, Lobuche, Changri Nup/Shar, Imja-Lhotse Shar, Ama Dablam
BBOX = [86.60, 27.83, 87.00, 28.12]   # lon_min, lat_min, lon_max, lat_max
CRS = 'EPSG:32645'                    # UTM 45N
S2_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

## 2. List good dates for every year
Cloud % is measured **inside our area only**, using the SCL band (classes 3 shadow, 8/9 cloud, 10 cirrus).
Only dates that cover >95% of the area and have <20% cloud are shown. October dates come first.

In [ ]:
def s2_items(start, end):
    return catalog.search(collections=['sentinel-2-l2a'], bbox=BBOX,
                          datetime=f'{start}/{end}').item_collection()

def candidates(year, max_cloud=20, min_cover=95):
    items = s2_items(f'{year}-10-01', f'{year}-11-30')
    if len(items) == 0:
        return pd.DataFrame()
    scl = load(items, bands=['SCL'], bbox=BBOX, crs=CRS, resolution=60,
               groupby='solar_day').SCL
    valid = scl > 0
    cloud = scl.isin([3, 8, 9, 10])
    snow  = scl == 11                      # rough hint only - check by eye
    cover = (valid.mean(('x', 'y')) * 100).values
    cc    = (cloud.sum(('x', 'y')) / valid.sum(('x', 'y')) * 100).values
    sn    = (snow.sum(('x', 'y')) / valid.sum(('x', 'y')) * 100).values
    df = pd.DataFrame({'date': [str(t)[:10] for t in scl.time.values],
                       'cloud_%': cc.round(1), 'cover_%': cover.round(1),
                       'snow_%_hint': sn.round(1)})
    df['month'] = df.date.str[5:7]
    df = df[(df['cover_%'] > min_cover) & (df['cloud_%'] < max_cloud)]
    return df.sort_values(['month', 'cloud_%']).drop(columns='month')

all_cands = {}
for y in range(2016, 2026):
    try:
        all_cands[y] = candidates(y)
    except Exception as e:
        print(y, 'ERROR:', e); all_cands[y] = pd.DataFrame()
    print(f'\n===== {y}: {len(all_cands[y])} usable dates =====')
    print(all_cands[y].head(8).to_string(index=False) if len(all_cands[y]) else 'none - tell the group')

**If a year shows "none":** re-run it with looser limits, e.g. `candidates(2016, max_cloud=30, min_cover=90)`.
Still nothing → try September / early December, and note it in the log.

`snow_%_hint` is from the automatic classifier; compare dates within a year (big jump = probably fresh snow) but always confirm by eye below.

## 3. Preview a date
Left: true colour. Right: SWIR composite (B11-B8-B4) – snow/ice is **cyan**, debris is brown/grey, clouds are white.
Fresh snow = cyan spread over valley floors and ridges outside the glaciers → mark *fresh snow = yes* and pick another date.

In [ ]:
def preview(date):
    items = s2_items(date, date)
    ds = load(items, bands=['B02', 'B03', 'B04', 'B08', 'B11'], bbox=BBOX,
              crs=CRS, resolution=60, groupby='solar_day').isel(time=0)
    def stretch(a, lo=0, hi=4000):
        return np.clip((a - lo) / (hi - lo), 0, 1)
    rgb  = np.dstack([stretch(ds[b].values.astype(float)) for b in ['B04', 'B03', 'B02']])
    swir = np.dstack([stretch(ds[b].values.astype(float), 0, 5000) for b in ['B11', 'B08', 'B04']])
    fig, ax = plt.subplots(1, 2, figsize=(16, 7))
    ax[0].imshow(rgb);  ax[0].set_title(f'{date} true colour')
    ax[1].imshow(swir); ax[1].set_title(f'{date} SWIR (snow/ice = cyan)')
    for a in ax: a.axis('off')
    plt.show()

preview('2020-10-15')   # <- change to a date from the list above

## 4. Fill in your chosen date for each year
Also record whether you saw fresh snow (Y/N) and any notes. This becomes the scene log.

In [ ]:
CHOSEN = {
    # year: ('YYYY-MM-DD', 'fresh snow Y/N', 'notes')
    2016: ('', 'N', ''),
    2017: ('', 'N', ''),
    2018: ('', 'N', ''),
    2019: ('', 'N', ''),
    2020: ('', 'N', ''),
    2021: ('', 'N', ''),
    2022: ('', 'N', ''),
    2023: ('', 'N', ''),
    2024: ('', 'N', ''),
    2025: ('', 'N', ''),
}

## 5. Export all chosen images (10 m, UTM 45N, cropped)
Each GeoTIFF has 7 bands: B02, B03, B04, B08, B11, B12, SCL.

**Offset fix:** images processed with baseline ≥ 04.00 (from 25 Jan 2022) have +1000 added to reflectance bands.
This cell removes it automatically so all years are comparable. SCL is not changed. Nodata stays 0.

In [ ]:
def export_s2(date):
    items = s2_items(date, date)
    baselines = sorted({it.properties.get('s2:processing_baseline', '00.00') for it in items})
    ds = load(items, bands=S2_BANDS + ['SCL'], bbox=BBOX, crs=CRS,
              resolution=10, groupby='solar_day').isel(time=0)
    if any(b >= '04.00' for b in baselines):
        for b in S2_BANDS:
            v = ds[b].astype('int32')
            ds[b] = xr.where(v > 0, np.clip(v - 1000, 1, None), 0).astype('uint16')
    arr = ds[S2_BANDS + ['SCL']].to_array('band').astype('uint16')
    arr = arr.rio.write_crs(CRS).rio.write_nodata(0)
    arr.attrs['long_name'] = tuple(S2_BANDS + ['SCL'])   # band names inside the GeoTIFF
    path = f"{OUT}/sentinel2/S2_{date.replace('-', '')}.tif"
    arr.rio.to_raster(path, compress='deflate')
    return path, baselines, [it.id for it in items]

log_rows = []
for y, (date, snow, notes) in CHOSEN.items():
    if not date:
        print(y, 'skipped (no date filled in)'); continue
    path, bl, ids = export_s2(date)
    c = all_cands.get(y, pd.DataFrame())
    cloud = c.loc[c.date == date, 'cloud_%'].values
    log_rows.append({'year': y, 'date': date, 'tile_ids': '; '.join(ids),
                     'processing_baseline': ', '.join(bl),
                     'cloud_%_aoi': float(cloud[0]) if len(cloud) else None,
                     'fresh_snow': snow, 'notes': notes, 'file': os.path.basename(path)})
    print(y, '->', path)

## 6. Copernicus DEM GLO-30 (elevation)
Exported at 30 m on the same UTM 45N grid. Part 2 will derive slope from it and resample to 10 m.

In [ ]:
dem_items = catalog.search(collections=['cop-dem-glo-30'], bbox=BBOX).item_collection()
dem = load(dem_items, bands=['data'], bbox=BBOX, crs=CRS, resolution=30,
           groupby='solar_day').data.max('time')
dem = dem.rio.write_crs(CRS).rio.write_nodata(-32767)
dem.rio.to_raster(f'{OUT}/dem/CopDEM_GLO30_UTM45N.tif', compress='deflate')
print('DEM range (m):', float(dem.min()), '-', float(dem.max()))   # expect roughly 3500-8800
dem.plot(figsize=(8, 6), cmap='terrain'); plt.title('Copernicus DEM'); plt.show()

## 7. Landsat 8/9 thermal band (for debris-covered ice)
One clear Oct–Nov date per year. Output is surface temperature in **Kelvin** (band `lwir11` = ST_B10), 30 m grid.
Landsat 9 exists only from late 2021; Landsat 8 covers all years.

In [ ]:
def ls_candidates(year):
    items = catalog.search(collections=['landsat-c2-l2'], bbox=BBOX,
                           datetime=f'{year}-10-01/{year}-11-30',
                           query={'platform': {'in': ['landsat-8', 'landsat-9']}}).item_collection()
    if len(items) == 0:
        return pd.DataFrame(), items
    qa = load(items, bands=['qa_pixel'], bbox=BBOX, crs=CRS, resolution=90,
              groupby='solar_day').qa_pixel
    valid = (qa & 1) == 0                              # bit 0 = fill
    cloud = ((qa >> 3) & 1) | ((qa >> 4) & 1)          # bit 3 cloud, bit 4 shadow
    cover = (valid.mean(('x', 'y')) * 100).values
    cc = ((cloud * valid).sum(('x', 'y')) / valid.sum(('x', 'y')) * 100).values
    df = pd.DataFrame({'date': [str(t)[:10] for t in qa.time.values],
                       'cloud_%': cc.round(1), 'cover_%': cover.round(1)})
    return df[df['cover_%'] > 90].sort_values('cloud_%'), items

def export_ls(date):
    items = catalog.search(collections=['landsat-c2-l2'], bbox=BBOX, datetime=f'{date}/{date}',
                           query={'platform': {'in': ['landsat-8', 'landsat-9']}}).item_collection()
    ds = load(items, bands=['lwir11'], bbox=BBOX, crs=CRS, resolution=30,
              groupby='solar_day').isel(time=0)
    st = ds.lwir11.astype('float32')
    st = xr.where(st > 0, st * 0.00341802 + 149.0, np.nan).astype('float32')   # Kelvin
    st = st.rio.write_crs(CRS).rio.write_nodata(np.nan)
    path = f"{OUT}/landsat_thermal/LST_{date.replace('-', '')}.tif"
    st.rio.to_raster(path, compress='deflate')
    return path

for y in range(2016, 2026):
    df, _ = ls_candidates(y)
    if len(df) == 0:
        print(y, 'no Landsat scene'); continue
    best = df.iloc[0]
    print(y, 'best:', best.date, f"cloud {best['cloud_%']}%")
    if best['cloud_%'] < 30:
        print('   ->', export_ls(best.date))
    else:
        print('   too cloudy, skipped - check manually')

## 8. Save the scene log
Also paste it into the shared Google Sheet.

In [ ]:
log = pd.DataFrame(log_rows)
log.to_csv(f'{OUT}/scene_log.csv', index=False)
log

## Still to download by hand (not on Planetary Computer)
- **RGI 7.0**, region 15 South Asia East: https://nsidc.org/data/nsidc-0770/versions/7 (Earthdata login) → put in `raw/rgi/`
- **Hugonnet dh/dt**, tiles for 27–28°N, 86–87°E, periods 2000–2019 and 2015–2019: https://www.theia-land.fr/en/blog/product/rate-of-glacier-elevation-changes-from-2000-to-2019/ → put in `raw/hugonnet/`

**Done when:** 10 S2 files + DEM + thermal files + RGI + Hugonnet are in Drive and the scene log is filled in.